In [1]:
import numpy as np
import pandas as pd

def download_iris_data(save_path="iris.data"):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
    col_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
    try:
        df = pd.read_csv(url, header=None, names=col_names)
        df.to_csv(save_path, index=False, header=False)
        print(f"数据集已通过pandas成功下载到：{save_path}")
    except Exception as e:
        print(f"下载失败：{e}")
        print("请检查网络，或手动下载数据集到代码同级目录")

download_iris_data()

class KMeans:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None

    def _euclidean_distance(self, X, centers):
        return np.sqrt(np.sum((X[:, np.newaxis, :] - centers) ** 2, axis=2))

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        n_samples, n_features = X.shape

        rng = np.random.default_rng(self.random_state)
        init_indices = rng.choice(n_samples, size=self.n_clusters, replace=False)
        self.cluster_centers_ = X[init_indices].copy()

        for _ in range(self.max_iter):
            distances = self._euclidean_distance(X, self.cluster_centers_)
            self.labels_ = np.argmin(distances, axis=1)

            new_centers = np.zeros((self.n_clusters, n_features), dtype=np.float64)
            for i in range(self.n_clusters):
                cluster_samples = X[self.labels_ == i]
                if cluster_samples.shape[0] == 0:
                    new_centers[i] = X[rng.choice(n_samples, size=1)]
                else:
                    new_centers[i] = np.mean(cluster_samples, axis=0)

            self.inertia_ = np.sum(np.min(distances ** 2, axis=1))
            center_diff = np.sum(np.abs(new_centers - self.cluster_centers_))
            if center_diff < self.tol:
                break

            self.cluster_centers_ = new_centers

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        distances = self._euclidean_distance(X, self.cluster_centers_)
        return np.argmin(distances, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

def load_iris_numpy(file_path="iris.data"):
    data = np.genfromtxt(file_path, delimiter=',', dtype=str)
    X = data[:, :-1].astype(np.float64)
    y_str = data[:, -1]
    unique_classes = np.unique(y_str)
    y_true = np.zeros(len(y_str), dtype=np.int32)
    for i, cls in enumerate(unique_classes):
        y_true[y_str == cls] = i
    return X, y_true, unique_classes

def standard_scaler_numpy(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    std[std == 0] = 1
    return (X - mean) / std

def clustering_accuracy_numpy(y_true, y_pred):
    n_classes = len(np.unique(y_true))
    conf_mat = np.zeros((n_classes, n_classes), dtype=np.int32)
    for t, p in zip(y_true, y_pred):
        conf_mat[t, p] += 1

    accuracy = 0
    for true_cls in range(n_classes):
        pred_cls = np.argmax(conf_mat[true_cls])
        accuracy += conf_mat[true_cls, pred_cls]
    accuracy /= len(y_true)
    return accuracy, None

if __name__ == "__main__":
    X, y_true, _ = load_iris_numpy()
    X_scaled = standard_scaler_numpy(X)
    
    kmeans = KMeans(n_clusters=3, random_state=42)
    y_pred = kmeans.fit_predict(X_scaled)
    
    print(f"簇中心：\n{kmeans.cluster_centers_}")
    print(f"惯性值：{kmeans.inertia_:.4f}")
    acc, _ = clustering_accuracy_numpy(y_true, y_pred)
    print(f"聚类准确率：{acc * 100:.2f}%")

数据集已通过pandas成功下载到：iris.data
簇中心：
[[-0.16840578 -0.97008147  0.25962078  0.17609756]
 [-1.00206653  0.89510445 -1.30297509 -1.25663117]
 [ 1.03359865  0.01388418  0.94369497  0.97226253]]
惯性值：142.1106
聚类准确率：85.33%
